#### 29.10.25, &copy; [Evhenii Kostin](https://github.com/DE123MasterProgram2025autumn/DE_Kostin), 2025

# Лабораторна робота №7: Основи роботи з Apache Spark у середовищі R (через sparklyr)


__Мета:__ _навчитися налаштовувати локальний кластер Apache Spark та виконувати базові операції з великими даними за допомогою R-інтерфейсу sparklyr. Освоїти основні етапи роботи з Spark: підключення, завантаження даних, аналіз через dplyr-синтаксис, моніторинг через веб-інтерфейс та коректне від’єднання._

### Варіант 2
#### Тема: Аналіз даних електронної комерції з використанням Apache Spark

In [1]:
# --- 0. ВСТАНОВЛЕННЯ ПАКЕТІВ ТА СИМУЛЯЦІЯ ---
# Встановлюємо sparklyr та data.table (для швидкої генерації файлу)
if (!require("sparklyr")) install.packages("sparklyr")
if (!require("data.table")) install.packages("data.table")
if (!require("dplyr")) install.packages("dplyr") # dplyr потрібен для синтаксису

library(sparklyr)
library(data.table)
library(dplyr)

# Створюємо папку для даних
dir.create("data", showWarnings = FALSE)

# --- Симуляція sales_2024.csv ---
message("Створення sales_2024.csv (500,000 рядків)...")
set.seed(123)
product_count <- 100
products_dt <- data.table(
  product_id = paste0("P", 1000:(1000 + product_count - 1)),
  category = sample(c("Electronics", "Clothing", "Home", "Books"), product_count, replace = TRUE),
  brand = sample(c("Brand_A", "Brand_B", "Brand_C"), product_count, replace = TRUE),
  price = round(runif(product_count, 10, 1000), 2)
)

n_rows <- 500000
sales_dt <- data.table(
  order_id = 1:n_rows,
  customer_id = paste0("C", sample(1:50000, n_rows, replace = TRUE)),
  product_id = sample(products_dt$product_id, n_rows, replace = TRUE),
  quantity = sample(1:5, n_rows, replace = TRUE),
  order_date = sample(
    seq(as.IDate("2024-01-01"), as.IDate("2024-12-31"), by = "day"),
    n_rows,
    replace = TRUE
  )
)

# Додаємо дані про продукти (ціна, категорія, бренд) до файлу
sales_full_dt <- merge(sales_dt, products_dt, by = "product_id")
fwrite(sales_full_dt, "data/sales_2024.csv")

message("--- ЕТАП 0 ЗАВЕРШЕНО. Файли створено. ---")

Loading required package: sparklyr

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘sparklyr’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘config’


Loading required package: data.table

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘sparklyr’


The following object is masked from ‘package:stats’:

    filter


Створення sales_2024.csv (500,000 рядків)...

--- ЕТАП 0 ЗАВЕРШЕНО. Файли створено. ---



### Етап 1: Встановлення та підключення
- Встановити Java 8, пакет sparklyr та Spark (версія 3.3.2) через spark_install().
- Підключитися до локального кластера Spark через spark_connect(master = "local").

In [3]:
# --- Етап 1: (Фінальна надійна версія, з ручним завантаженням ngrok) ---

# 0. Завантажуємо httr ТА SPARKLYR
if (!require("httr")) install.packages("httr")
if (!require("sparklyr")) install.packages("sparklyr")
if (!require("dplyr")) install.packages("dplyr")

library(httr)
library(sparklyr) # <--- 💡 ОСЬ ВИПРАВЛЕННЯ (додано)
library(dplyr)    # <--- 💡 І ЦЕ ТАКОЖ (знадобиться пізніше)

# --- КРОК A: Примусово від'єднуємо ВСІ старі сесії ---
message("Від'єднуємо всі старі сесії Spark...")
try(spark_disconnect_all(), silent = TRUE)
Sys.sleep(2) # Даємо 2 секунди, щоб порти звільнилися

# --- КРОК B: Встановлюємо Java (з init.ipynb) ---
Sys.setenv(JAVA_HOME = "/usr/lib/jvm/java-11-openjdk-amd64")
message(paste("Встановлено JAVA_HOME:", Sys.getenv("JAVA_HOME")))

# --- КРОК C: Перевіряємо Spark (з init.ipynb) ---
if (nrow(spark_installed_versions()) == 0) {
  message("Встановлення Apache Spark 3.3.2...")
  spark_install(version = "3.3.2")
} else {
  message("Apache Spark вже встановлено.")
}

# --- КРОК D: Ручне встановлення NGROK (вирішення проблеми) ---
if (!file.exists("ngrok")) {
  message("NGROK не знайдено. Завантажуємо вручну...")
  # Завантажуємо архів
  download.file("https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz", "ngrok.tgz")
  # Розпаковуємо архів (tar -xvzf ...)
  system("tar -xvzf ngrok.tgz")
  # Робимо файл виконуваним
  system("chmod +x ngrok")
  message("NGROK успішно завантажено та розпаковано.")
  
  # Додаємо Authtoken
  message("Додаємо Authtoken...")
  # ❗️ Вставте ваш токен сюди, якщо 'my_authtoken' ще не визначено
  if (!exists("my_authtoken")) {
      my_authtoken <- "35GT3Td5mRfzENr5yHxLeCdUCGI_3uSNAs8QzzgDfCjn67v8N"
  }
  system(paste("./ngrok config add-authtoken", my_authtoken))
} else {
  message("NGROK вже завантажено.")
}

# --- КРОК E: Підключаємось (створюємо НОВУ сесію) ---
sc <- spark_connect(master = "local")
message("Spark підключено (НОВА сесія). Запускаємо ngrok...")

# --- КРОК F: Запускаємо ngrok з локального файлу ---
system("killall ngrok > /dev/null 2>&1") # Вбиваємо старі процеси
Sys.sleep(1)
    
# Запускаємо з локальної папки (./ngrok)
system("./ngrok http 4040 --log=stdout > ngrok.log &")
Sys.sleep(3) # Даємо ngrok 3 секунди на запуск
    
# --- КРОК G: Отримуємо URL ---
tryCatch({
  resp <- GET("http://localhost:4041/api/tunnels")
  content <- content(resp, "parsed")
  public_url <- content$tunnels[[1]]$public_url
  
  message("\n--- 🚀 ВЕБ-ІНТЕРФЕЙС SPARK ГОТОВИЙ! ---")
  message(paste("Ваша зовнішня веб-адреса:", public_url))
  message("Натисніть на це посилання, щоб відкрити Spark UI.")
  
}, error = function(e) {
  message("\n--- ❗ ПОМИЛКА: Не вдалося отримати URL від ngrok. ---")
  message("Перегляньте лог ngrok, виконавши: system('cat ngrok.log')")
})

message("\n--- ЕТАП 1 ЗАВЕРШЕNO. ---")

Loading required package: sparklyr

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘sparklyr’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘config’


Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘sparklyr’


The following object is masked from ‘package:stats’:

    filter


Від'єднуємо всі старі сесії Spark...



[1] 0

Встановлено JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64

Встановлення Apache Spark 3.3.2...

NGROK не знайдено. Завантажуємо вручну...

NGROK успішно завантажено та розпаковано.

Додаємо Authtoken...

Spark підключено (НОВА сесія). Запускаємо ngrok...

Warning message in system("killall ngrok > /dev/null 2>&1"):
“error in running command”

--- 🚀 ВЕБ-ІНТЕРФЕЙС SPARK ГОТОВИЙ! ---

Ваша зовнішня веб-адреса: https://paz-inseverable-noemi.ngrok-free.dev

Натисніть на це посилання, щоб відкрити Spark UI.


--- ЕТАП 1 ЗАВЕРШЕNO. ---



### Етап 2: Завантаження даних
- Завантажити файл sales_2024.csv у Spark через copy_to() або spark_read_csv().
- Переконатися, що таблиця доступна у Spark (використовуйте glimpse() або count()).

In [4]:
# --- Етап 2: Завантаження даних ---
message("--- ЕТАП 2: ЗАВАНТАЖЕННЯ ДАНИХ У SPARK ---")

# 1. Завантажуємо файл у Spark
# Ми не читаємо його в R, ми кажемо Spark прочитати його
sales_tbl <- spark_read_csv(
  sc, 
  name = "sales_2024", # Назва таблиці у Spark
  path = "data/sales_2024.csv"
)

# 2. Перевіряємо, що дані завантажились
message("\nСтруктура даних у Spark (glimpse):")
glimpse(sales_tbl)

message("\nПідрахунок рядків у Spark (count):")
print(count(sales_tbl))

message("--- ЕТАП 2 ЗАВЕРШЕНО. ---")

--- ЕТАП 2: ЗАВАНТАЖЕННЯ ДАНИХ У SPARK ---


Структура даних у Spark (glimpse):

Rows: ??
Columns: 8
Database: spark_connection
$ product_id  <chr> "P1000", "P1000", "P1000", "P1000", "P1000", "P1000", "P10…
$ order_id    <int> 8, 95, 580, 684, 787, 1224, 1287, 1350, 1594, 1723, 1874, …
$ customer_id <chr> "C12637", "C7880", "C31566", "C47086", "C30552", "C26921",…
$ quantity    <int> 5, 1, 1, 1, 4, 1, 3, 3, 4, 4, 2, 2, 3, 2, 4, 4, 2, 2, 1, 5…
$ order_date  <dttm> 2024-12-01, 2024-04-12, 2024-04-23, 2024-11-06, 2024-01-3…
$ category    <chr> "Home", "Home", "Home", "Home", "Home", "Home", "Home", "H…
$ brand       <chr> "Brand_A", "Brand_A", "Brand_A", "Brand_A", "Brand_A", "Br…
$ price       <dbl> 551.36, 551.36, 551.36, 551.36, 551.36, 551.36, 551.36, 55…

Підрахунок рядків у Spark (count):

# Source:   SQL [?? x 1]
# Database: spark_connection
       n
   <dbl>
1 500000
--- ЕТАП 2 ЗАВЕРШЕНО. ---



### Етап 3: Аналіз через dplyr-інтерфейс
Виконати наступні операції у Spark, використовуючи синтаксис dplyr:
- Додати стовпець revenue = quantity * price.
- Відфільтрувати замовлення з revenue > 1000.
- Обчислити загальний дохід за кожною комбінацією category та brand.
- Відсортувати результат за спаданням revenue.
- Вибрати топ-5 категорій з найвищим доходом.

In [5]:
# --- Етап 3: Аналіз через dplyr-інтерфейс ---
message("--- ЕТАП 3: АНАЛІЗ ДАНИХ У SPARK ---")

# Виконуємо всі операції у Spark
top_5_analysis <- sales_tbl %>%
  # - Додаємо стовпець revenue = quantity * price
  mutate(revenue = quantity * price) %>%
  
  # - Відфільтруємо замовлення з revenue > 1000
  filter(revenue > 1000) %>%
  
  # - Обчисліть загальний дохід за кожною комбінацією category та brand
  group_by(category, brand) %>%
  summarise(
    total_revenue = sum(revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  
  # - Відсортуйте результат за спаданням revenue
  arrange(desc(total_revenue)) %>%
  
  # - Виберіть топ-5 категорій
  head(5) # `head(5)` перетворюється на `LIMIT 5` у Spark SQL

message("План запиту (аналіз) створено. Дані ще не в R.")
print(top_5_analysis)

message("--- ЕТАП 3 ЗАВЕРШЕНО. ---")

--- ЕТАП 3: АНАЛІЗ ДАНИХ У SPARK ---

План запиту (аналіз) створено. Дані ще не в R.

# Source:     SQL [?? x 3]
# Database:   spark_connection
# Ordered by: desc(total_revenue)
  category    brand   total_revenue
  <chr>       <chr>           <dbl>
1 Electronics Brand_C     83266799.
2 Home        Brand_C     77962957.
3 Clothing    Brand_B     76880565.
4 Electronics Brand_B     75870391.
5 Home        Brand_B     69868872.
--- ЕТАП 3 ЗАВЕРШЕНО. ---



### Етап 4: Отримання результатів у R
- Завантажити результат у R через collect().
- Переконатися, що результат — це data.frame з 5 рядками.

In [6]:
# --- Етап 4: Отримання результатів у R ---
message("--- ЕТАП 4: ОТРИМАННЯ РЕЗУЛЬТАТІВ У R ---")

# 1. Завантажуємо результат у R
# Ця команда змушує Spark виконати весь аналіз з Етапу 3
top_5_results_df <- collect(top_5_analysis)

# 2. Перевіряємо результат
message(paste("\nРезультат отримано. Клас об'єкта:", class(top_5_results_df)[1]))
message(paste("Кількість рядків:", nrow(top_5_results_df)))

print(top_5_results_df)

message("--- ЕТАП 4 ЗАВЕРШЕНО. ---")

--- ЕТАП 4: ОТРИМАННЯ РЕЗУЛЬТАТІВ У R ---


Результат отримано. Клас об'єкта: tbl_df

Кількість рядків: 5

# A tibble: 5 × 3
  category    brand   total_revenue
  <chr>       <chr>           <dbl>
1 Electronics Brand_C     83266799.
2 Home        Brand_C     77962957.
3 Clothing    Brand_B     76880565.
4 Electronics Brand_B     75870391.
5 Home        Brand_B     69868872.
--- ЕТАП 4 ЗАВЕРШЕНО. ---



### Етап 5: Моніторинг через веб-інтерфейс
- Відкрити веб-інтерфейс Spark через spark_web(sc).
- Перевірити:
- - чи з’явився датафрейм у вкладці Storage
- - чи виконався запит у вкладці Jobs
- - скільки пам’яті використовується (вкладка Executors).

In [7]:
# --- Етап 5: Моніторинг через веб-інтерфейс ---
message("--- ЕТАП 5: МОНІТОРИНГ ЧЕРЕЗ ВЕБ-ІНТЕРФЕЙС ---")

# 1. Відкриваємо веб-інтерфейс
# Ця команда може не спрацювати, але порт вже відкритий
try(spark_web(sc), silent = TRUE)

message("\n--- ІНСТРУКЦІЇ ДЛЯ DEEPNOTE ---")
message("Тепер вручну натисніть на посилання 'Port 8080' у панелі 'Machine' (ліворуч).")
message("Це має бути веб-інтерфейс Spark.")

message("--- ЕТАП 5 ЗАВЕРШЕНО. ---")

--- ЕТАП 5: МОНІТОРИНГ ЧЕРЕЗ ВЕБ-ІНТЕРФЕЙС ---


--- ІНСТРУКЦІЇ ДЛЯ DEEPNOTE ---

Тепер вручну натисніть на посилання 'Port 8080' у панелі 'Machine' (ліворуч).

Це має бути веб-інтерфейс Spark.

--- ЕТАП 5 ЗАВЕРШЕНО. ---



### Етап 6: Від’єднання
- Коректно від’єднатися від кластера через spark_disconnect(sc).

In [8]:
# --- Етап 6: Від’єднання ---
message("--- ЕТАП 6: ВІД'ЄДНАННЯ ВІД КЛАСТЕРА ---")

spark_disconnect(sc)

message("Успішно від'єднано від Spark.")
message("--- ЛАБОРАТОРНУ РОБОТУ ЗАВЕРШЕНО. ---")

--- ЕТАП 6: ВІД'ЄДНАННЯ ВІД КЛАСТЕРА ---

Успішно від'єднано від Spark.

--- ЛАБОРАТОРНУ РОБОТУ ЗАВЕРШЕНО. ---



<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6a083947-94ba-475d-8730-27cff0574f54' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>